# Day 30 — Automated Invoice Processing System

This notebook processes invoice data from CSV and demonstrates automated extraction from text-based PDF invoices, invoice consolidation, total calculation, overdue detection, summary generation, and CSV report export.




In [1]:
from pathlib import Path
import re
import pandas as pd
import numpy as np

# ============================================================
#  PROJECT PATHS
# ============================================================

PROJECT_DIR = Path.cwd()

# Find the project root even if Jupyter was launched elsewhere
candidates = [
    PROJECT_DIR,
    PROJECT_DIR / "Day 30",
    PROJECT_DIR.parent,
]

PROJECT_ROOT = PROJECT_DIR

for candidate in candidates:
    if (candidate / "sample_invoices.csv").exists() or (candidate / "data" / "sample_invoices.csv").exists():
        PROJECT_ROOT = candidate
        break

REPORTS = PROJECT_ROOT / "reports"
REPORTS.mkdir(parents=True, exist_ok=True)

# Your screenshot shows sample_invoices.csv directly inside Day 30.
# This also supports the cleaner data/sample_invoices.csv layout.
csv_candidates = [
    PROJECT_ROOT / "sample_invoices.csv",
    PROJECT_ROOT / "data" / "sample_invoices.csv",
]

CSV_PATH = next((p for p in csv_candidates if p.exists()), None)

print("=" * 70)
print(" AUTOMATED INVOICE PROCESSING SYSTEM")
print("=" * 70)
print(f" Project folder : {PROJECT_ROOT}")
print(f" Reports folder : {REPORTS}")

if CSV_PATH:
    print(f" Sample CSV found: {CSV_PATH}")
else:
    print(" Sample CSV not found.")
    print("Expected either:")
    print("   Day 30/sample_invoices.csv")
    print("   Day 30/data/sample_invoices.csv")


 AUTOMATED INVOICE PROCESSING SYSTEM
 Project folder : c:\Users\itssh\OneDrive\Documents\GitHub\rftinternship\RFTinternship\Day 30
 Reports folder : c:\Users\itssh\OneDrive\Documents\GitHub\rftinternship\RFTinternship\Day 30\reports
 Sample CSV found: c:\Users\itssh\OneDrive\Documents\GitHub\rftinternship\RFTinternship\Day 30\sample_invoices.csv


##  Dataset Submission Block

You can use the included sample CSV or enter the path to your own invoice CSV. The required fields are `Invoice Number`, `Customer Name`, and `Invoice Date`.

In [2]:
# ============================================================
#  DATASET SUBMISSION
# ============================================================

print("Choose your invoice dataset:")
print("1️  Use the included sample CSV")
print("2️  Use your own CSV file")

choice = input("Enter choice [1]: ").strip() or "1"

if choice == "1":
    if CSV_PATH is None:
        raise FileNotFoundError(
            "sample_invoices.csv was not found. "
            "Place it in the Day 30 folder or Day 30/data/."
        )
    INPUT_PATH = CSV_PATH

elif choice == "2":
    user_path = input("Enter the full path to your CSV: ").strip().strip('"')
    INPUT_PATH = Path(user_path).expanduser()

    if not INPUT_PATH.exists():
        raise FileNotFoundError(f"File not found: {INPUT_PATH}")

else:
    raise ValueError("Please enter 1 or 2.")

# ------------------------------------------------------------
# LOAD
# ------------------------------------------------------------

df = pd.read_csv(INPUT_PATH)

required = {
    "Invoice Number",
    "Customer Name",
    "Invoice Date",
}

missing = required - set(df.columns)

if missing:
    raise ValueError(
        "Missing required columns: " + ", ".join(sorted(missing))
    )

# Add optional columns when absent
optional_columns = [
    "Customer Email",
    "Due Date",
    "Item",
    "Quantity",
    "Unit Price",
    "Amount",
    "Tax",
]

for column in optional_columns:
    if column not in df.columns:
        df[column] = ""

# Clean dates
df["Invoice Date"] = pd.to_datetime(
    df["Invoice Date"],
    errors="coerce",
    dayfirst=True
)

df["Due Date"] = pd.to_datetime(
    df["Due Date"],
    errors="coerce",
    dayfirst=True
)

# Clean numeric values
def clean_number(value):
    text = str(value).replace(",", "")
    text = re.sub(r"[₹$£€]", "", text)
    text = re.sub(r"[^0-9.\-]", "", text)
    try:
        return float(text)
    except ValueError:
        return 0.0

df["Quantity"] = pd.to_numeric(
    df["Quantity"],
    errors="coerce"
).fillna(1)

df["Unit Price"] = df["Unit Price"].apply(clean_number)
df["Amount"] = df["Amount"].apply(clean_number)
df["Tax"] = df["Tax"].apply(clean_number)

# If Amount is not supplied, calculate it
if df["Amount"].eq(0).all():
    df["Amount"] = (
        df["Quantity"] * df["Unit Price"] + df["Tax"]
    )

df["Invoice Number"] = df["Invoice Number"].astype(str).str.strip()
df["Customer Name"] = df["Customer Name"].astype(str).str.strip()
df["Customer Email"] = df["Customer Email"].astype(str).str.strip()
df["Item"] = df["Item"].astype(str).str.strip()

# Remove invalid essential records
df = df.dropna(
    subset=["Invoice Number", "Customer Name", "Invoice Date"]
).copy()

print("\n" + "=" * 70)
print(" DATASET LOADED SUCCESSFULLY")
print("=" * 70)
print(f" File       : {INPUT_PATH.name}")
print(f" Line items : {len(df):,}")
print(f" Columns    : {len(df.columns)}")

display(df.head(10))


Choose your invoice dataset:
1️  Use the included sample CSV
2️  Use your own CSV file



 DATASET LOADED SUCCESSFULLY
 File       : sample_invoices.csv
 Line items : 4
 Columns    : 10


C:\Users\itssh\AppData\Local\Temp\ipykernel_7936\2545272682.py:70: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df["Due Date"] = pd.to_datetime(


,Invoice Number,Customer Name,Customer Email,Invoice Date,Due Date,Item,Quantity,Unit Price,Amount,Tax
0,INV-1001,Acme Technologies,accounts@acme.example,2026-05-05,2026-05-20,Website Development,1,45000.0,45000.0,0.0
1,INV-1002,Bright Retail,finance@bright.example,2026-12-06,2026-06-27,Analytics Dashboard,1,28000.0,28000.0,0.0
2,INV-1003,Northstar Labs,billing@northstar.example,2026-02-07,2026-07-17,Data Cleaning,1,18000.0,18000.0,0.0
4,INV-1005,Vertex Media,payments@vertex.example,2026-04-08,2026-08-19,Social Media Analytics,1,22000.0,22000.0,0.0


##  Invoice & Item Processing

In [3]:
# ============================================================
# CONSOLIDATE MULTIPLE ITEM ROWS INTO ONE INVOICE
# ============================================================

rows = []

for invoice_number, group in df.groupby(
    "Invoice Number",
    sort=False
):
    first = group.iloc[0]

    item_names = [
        str(item).strip()
        for item in group["Item"]
        if str(item).strip()
    ]

    calculated_total = float(group["Amount"].sum())

    rows.append({
        "Invoice Number": invoice_number,
        "Customer Name": first["Customer Name"],
        "Customer Email": first["Customer Email"],
        "Invoice Date": first["Invoice Date"],
        "Due Date": first["Due Date"],
        "Items": "; ".join(item_names),
        "Item Count": len(item_names),
        "Calculated Total": calculated_total,
        "Invoice Total": calculated_total,
        "Source File": INPUT_PATH.name,
        "Source Type": "CSV",
    })

invoices = pd.DataFrame(rows)

print(f" Consolidated {len(invoices):,} invoice(s)")
display(invoices)


 Consolidated 4 invoice(s)


,Invoice Number,Customer Name,Customer Email,Invoice Date,Due Date,Items,Item Count,Calculated Total,Invoice Total,Source File,Source Type
0,INV-1001,Acme Technologies,accounts@acme.example,2026-05-05,2026-05-20,Website Development,1,45000.0,45000.0,sample_invoices.csv,CSV
1,INV-1002,Bright Retail,finance@bright.example,2026-12-06,2026-06-27,Analytics Dashboard,1,28000.0,28000.0,sample_invoices.csv,CSV
2,INV-1003,Northstar Labs,billing@northstar.example,2026-02-07,2026-07-17,Data Cleaning,1,18000.0,18000.0,sample_invoices.csv,CSV
3,INV-1005,Vertex Media,payments@vertex.example,2026-04-08,2026-08-19,Social Media Analytics,1,22000.0,22000.0,sample_invoices.csv,CSV


##  Identify Overdue Invoices

In [4]:
# ============================================================
#  OVERDUE ANALYSIS
# ============================================================

# Change this date if you want to test another reporting date.
AS_OF = pd.Timestamp.today().normalize()

invoices["Due Date"] = pd.to_datetime(
    invoices["Due Date"],
    errors="coerce"
)

invoices["Days Overdue"] = (
    AS_OF - invoices["Due Date"]
).dt.days.fillna(0).clip(lower=0)

invoices["Payment Status"] = np.where(
    invoices["Due Date"].notna()
    & (invoices["Due Date"] < AS_OF),
    "Overdue",
    "Current"
)

overdue = invoices[
    invoices["Payment Status"] == "Overdue"
].copy()

print(f" Checked as of: {AS_OF.date()}")
print(f" Overdue invoices: {len(overdue):,}")
print(f" Overdue amount: ₹{overdue['Invoice Total'].sum():,.2f}")

display(overdue)


 Checked as of: 2026-09-04
 Overdue invoices: 4
 Overdue amount: ₹113,000.00


,Invoice Number,Customer Name,Customer Email,Invoice Date,Due Date,Items,Item Count,Calculated Total,Invoice Total,Source File,Source Type,Days Overdue,Payment Status
0,INV-1001,Acme Technologies,accounts@acme.example,2026-05-05,2026-05-20,Website Development,1,45000.0,45000.0,sample_invoices.csv,CSV,107,Overdue
1,INV-1002,Bright Retail,finance@bright.example,2026-12-06,2026-06-27,Analytics Dashboard,1,28000.0,28000.0,sample_invoices.csv,CSV,69,Overdue
2,INV-1003,Northstar Labs,billing@northstar.example,2026-02-07,2026-07-17,Data Cleaning,1,18000.0,18000.0,sample_invoices.csv,CSV,49,Overdue
3,INV-1005,Vertex Media,payments@vertex.example,2026-04-08,2026-08-19,Social Media Analytics,1,22000.0,22000.0,sample_invoices.csv,CSV,16,Overdue


##  Automated Summary Report

In [5]:
summary = pd.DataFrame([{
    "Total Invoices": len(invoices),
    "Total Amount": invoices["Invoice Total"].sum(),
    "Average Invoice": invoices["Invoice Total"].mean(),
    "Overdue Invoices": len(overdue),
    "Overdue Amount": overdue["Invoice Total"].sum(),
    "Unique Customers": invoices["Customer Name"].nunique(),
}])

display(summary.style.format({
    "Total Amount": "₹{:,.2f}",
    "Average Invoice": "₹{:,.2f}",
    "Overdue Amount": "₹{:,.2f}",
}))


,Total Invoices,Total Amount,Average Invoice,Overdue Invoices,Overdue Amount,Unique Customers
0,4,"₹113,000.00","₹28,250.00",4,"₹113,000.00",4


##  Export Final Reports

In [6]:
# ============================================================
#  EXPORT REPORTS
# ============================================================

consolidated_path = REPORTS / "consolidated_invoice_report.csv"
overdue_path = REPORTS / "overdue_invoices.csv"
summary_path = REPORTS / "invoice_summary_report.csv"

invoices.to_csv(consolidated_path, index=False)
overdue.to_csv(overdue_path, index=False)
summary.to_csv(summary_path, index=False)

print(" Reports exported successfully:")
print(f"    {consolidated_path}")
print(f"    {overdue_path}")
print(f"    {summary_path}")


 Reports exported successfully:
    c:\Users\itssh\OneDrive\Documents\GitHub\rftinternship\RFTinternship\Day 30\reports\consolidated_invoice_report.csv
    c:\Users\itssh\OneDrive\Documents\GitHub\rftinternship\RFTinternship\Day 30\reports\overdue_invoices.csv
    c:\Users\itssh\OneDrive\Documents\GitHub\rftinternship\RFTinternship\Day 30\reports\invoice_summary_report.csv


##  PDF Invoice Extraction — Bonus

The following section extracts text from text-based PDF invoices using `pypdf`. It looks for invoice number, customer, email, invoice date, due date, item rows and total. Scanned/image-only PDFs require OCR and are not handled by this lightweight parser.

In [7]:
# ============================================================
# PDF EXTRACTION
# ============================================================

try:
    from pypdf import PdfReader

    invoice_folder = PROJECT_ROOT / "invoices"
    pdf_files = sorted(invoice_folder.glob("*.pdf"))

    if not pdf_files:
        print(" No PDF files found in invoices/.")
    else:
        print(f" Found {len(pdf_files)} PDF invoice(s)")

        pdf_results = []

        def extract_label(patterns, text):
            for pattern in patterns:
                match = re.search(pattern, text, re.IGNORECASE | re.MULTILINE)
                if match:
                    return match.group(1).strip()
            return ""

        for pdf_path in pdf_files:
            reader = PdfReader(str(pdf_path))
            text = "\n".join(
                page.extract_text() or ""
                for page in reader.pages
            )

            invoice_number = extract_label([
                r"Invoice\s*(?:No|Number|#)\s*[:\-]?\s*([A-Z0-9\-/]+)",
                r"INV(?:OICE)?\s*[:#\-]?\s*([A-Z0-9\-/]+)",
            ], text)

            customer = extract_label([
                r"Customer\s*Name\s*[:\-]\s*(.+)",
                r"Bill\s*To\s*[:\-]\s*(.+)",
                r"Customer\s*[:\-]\s*(.+)",
            ], text)

            email_match = re.search(
                r"[\w.+-]+@[\w-]+\.[\w.-]+",
                text
            )

            email = email_match.group(0) if email_match else ""

            invoice_date_raw = extract_label([
                r"Invoice\s*Date\s*[:\-]?\s*([^\n]+)"
            ], text)

            due_date_raw = extract_label([
                r"Due\s*Date\s*[:\-]?\s*([^\n]+)"
            ], text)

            invoice_date = pd.to_datetime(
                invoice_date_raw,
                errors="coerce",
                dayfirst=True
            )

            due_date = pd.to_datetime(
                due_date_raw,
                errors="coerce",
                dayfirst=True
            )

            totals = re.findall(
                r"(?:Grand\s+)?Total\s*[:\-]?\s*[₹$]?\s*([\d,]+(?:\.\d+)?)",
                text,
                re.IGNORECASE
            )

            total = clean_number(totals[-1]) if totals else 0.0

            pdf_results.append({
                "Invoice Number": invoice_number,
                "Customer Name": customer,
                "Customer Email": email,
                "Invoice Date": invoice_date,
                "Due Date": due_date,
                "Invoice Total": total,
                "Source File": pdf_path.name,
                "Source Type": "PDF",
            })

        pdf_invoices = pd.DataFrame(pdf_results)

        print("\n PDF extraction completed")
        display(pdf_invoices)

except ImportError:
    print(" pypdf is not installed.")
    print("Install it with: pip install pypdf")


 Found 3 PDF invoice(s)

 PDF extraction completed


,Invoice Number,Customer Name,Customer Email,Invoice Date,Due Date,Invoice Total,Source File,Source Type
0,INV-PDF-001,Acme Technologies,accounts@acme.example,2026-08-05,2026-08-20,57000.0,sample_invoice_001.pdf,PDF
1,INV-PDF-002,Bright Retail,finance@bright.example,2026-08-12,2026-08-27,42000.0,sample_invoice_002.pdf,PDF
2,INV-PDF-003,Northstar Labs,billing@northstar.example,2026-08-22,2026-09-01,41000.0,sample_invoice_003.pdf,PDF
